In [3]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

In [10]:
data = fetch_california_housing()
X, y = data.data, data.target

In [11]:
print(X.shape)
print(y.shape)

(20640, 8)
(20640,)


In [12]:
from sklearn.ensemble import VotingRegressor

In [17]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score

In [19]:
lr = LinearRegression()
dt = DecisionTreeRegressor()
svr = make_pipeline(StandardScaler(), SVR())

In [20]:
estimators = [('lr', lr), ('dt', dt), ('svr', svr)]

In [21]:
print("--- Base Model Scores ---")
for name, estimator in estimators:
    scores = cross_val_score(estimator, X, y, scoring='r2', cv=10)
    print(f"{name}: {np.round(np.mean(scores), 2)}")

--- Base Model Scores ---
lr: 0.51
dt: 0.23
svr: 0.65


In [22]:
print("\n--- Voting Regressor (Uniform) ---")
vr = VotingRegressor(estimators)
scores = cross_val_score(vr, X, y, scoring='r2', cv=10)
print(f"Voting Regressor: {np.round(np.mean(scores), 2)}")


--- Voting Regressor (Uniform) ---
Voting Regressor: 0.62


In [23]:
print("\n--- Hyperparameter Tuning Weights ---")
best_score = -np.inf
best_weights = None

for i in range(1, 4):
    for j in range(1, 4):
        for k in range(1, 4):
            vr = VotingRegressor(estimators, weights=[i, j, k])
            scores = cross_val_score(vr, X, y, scoring='r2', cv=10)
            mean_score = np.round(np.mean(scores), 2)
            print(f"For i={i}, j={j}, k={k}: {mean_score}")
            
            if mean_score > best_score:
                best_score = mean_score
                best_weights = [i, j, k]

print(f"\nBest Weights: {best_weights} with R2 Score: {best_score}")


--- Hyperparameter Tuning Weights ---
For i=1, j=1, k=1: 0.62
For i=1, j=1, k=2: 0.64
For i=1, j=1, k=3: 0.64
For i=1, j=2, k=1: 0.57
For i=1, j=2, k=2: 0.61
For i=1, j=2, k=3: 0.63
For i=1, j=3, k=1: 0.52
For i=1, j=3, k=2: 0.57
For i=1, j=3, k=3: 0.6
For i=2, j=1, k=1: 0.61
For i=2, j=1, k=2: 0.63
For i=2, j=1, k=3: 0.64
For i=2, j=2, k=1: 0.59
For i=2, j=2, k=2: 0.61
For i=2, j=2, k=3: 0.63
For i=2, j=3, k=1: 0.55
For i=2, j=3, k=2: 0.59
For i=2, j=3, k=3: 0.61
For i=3, j=1, k=1: 0.6
For i=3, j=1, k=2: 0.62
For i=3, j=1, k=3: 0.63
For i=3, j=2, k=1: 0.59
For i=3, j=2, k=2: 0.61
For i=3, j=2, k=3: 0.63
For i=3, j=3, k=1: 0.57
For i=3, j=3, k=2: 0.6
For i=3, j=3, k=3: 0.62

Best Weights: [1, 1, 2] with R2 Score: 0.64


In [24]:
print("\n--- Homogeneous Decision Tree Ensemble ---")
dt1 = DecisionTreeRegressor(max_depth=1)
dt2 = DecisionTreeRegressor(max_depth=3)
dt3 = DecisionTreeRegressor(max_depth=5)
dt4 = DecisionTreeRegressor(max_depth=7)
dt5 = DecisionTreeRegressor(max_depth=None)

dt_estimators = [('dt1', dt1), ('dt2', dt2), ('dt3', dt3), ('dt4', dt4), ('dt5', dt5)]

for name, estimator in dt_estimators:
    scores = cross_val_score(estimator, X, y, scoring='r2', cv=10)
    print(f"{name}: {np.round(np.mean(scores), 2)}")

vr_dt = VotingRegressor(dt_estimators)
vr_scores = cross_val_score(vr_dt, X, y, scoring='r2', cv=10)
print(f"Voting Regressor (Trees): {np.round(np.mean(vr_scores), 2)}")


--- Homogeneous Decision Tree Ensemble ---
dt1: 0.13
dt2: 0.36
dt3: 0.43
dt4: 0.47
dt5: 0.24
Voting Regressor (Trees): 0.5
